In [2]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

from vip_slap2_analysis.glutamate import summary as gs

sns.set()
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\andrew.shelton\AppData\Local\Temp\ipykernel_25752\1541214560.py:20: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
%matplotlib notebook

In [5]:
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [6]:
target_mice = [
    803496,804730,804733,810196,
    809047,803121,
    826033,834788,838410]

In [7]:
for mouse in target_mice:
    try:
        session_paths = [Path(p) for p in session_df[(session_df['subject_id']==mouse)
                                                     &(session_df['session_type']!='expression_check')]['session_dir'].values]
    except:
        print(f'No session path for {session_df[(session_df["subject_id"]==mouse)]["session_id"].values}')
    print(mouse)
    for sess in session_paths:
        summary_paths = glob.glob(os.path.join(sess, '**', 'SummaryLoCo**.mat'), recursive=True) 
    
        if len(summary_paths) == 0:
            print(f'No Summary_LoCo.mat found in {sess}')
            print('')
            continue

        summary_path = max(summary_paths, key=os.path.getmtime)
        exp = gs.GlutamateSummary(summary_path)
        n_trials = exp.n_trials
        print(f'Successfully loaded data from {sess.stem}')
        dmd1_valid_trials = exp.valid_trials[0]
        dmd2_valid_trials = exp.valid_trials[1]
        
        dmd1_vt_pct = len(dmd1_valid_trials)/n_trials
        dmd2_vt_pct = len(dmd2_valid_trials)/n_trials
        
        dmd_pct = [dmd1_vt_pct,dmd2_vt_pct]
        
        for i,dmd in enumerate([1,2]):
            print(f'% valid trials on DMD{dmd}: {dmd_pct[i]*100:.4}% with {exp.n_synapses[i]} synapses recorded')
        print('')
    print('')
#         print(f'Most recent Summary_LoCo.mat: {most_recent_summary}')
#         print('')

803496
Successfully loaded data from 2025-07-25_803496
% valid trials on DMD1: 100.0% with 92 synapses recorded
% valid trials on DMD2: 100.0% with 77 synapses recorded

Successfully loaded data from 2025-07-28_803496
% valid trials on DMD1: 100.0% with 94 synapses recorded
% valid trials on DMD2: 98.39% with 53 synapses recorded

Successfully loaded data from 2025-07-29_803496
% valid trials on DMD1: 100.0% with 83 synapses recorded
% valid trials on DMD2: 100.0% with 50 synapses recorded

Successfully loaded data from 2025-07-30_803496
% valid trials on DMD1: 100.0% with 104 synapses recorded
% valid trials on DMD2: 100.0% with 67 synapses recorded

Successfully loaded data from 2025-07-31_803496
% valid trials on DMD1: 100.0% with 68 synapses recorded
% valid trials on DMD2: 100.0% with 49 synapses recorded

Successfully loaded data from 2025-08-01_803496
% valid trials on DMD1: 95.16% with 85 synapses recorded
% valid trials on DMD2: 100.0% with 64 synapses recorded


804730
Succes

Successfully loaded data from 834788_2026-03-20_12-44-00
% valid trials on DMD1: 70.98% with 34 synapses recorded
% valid trials on DMD2: 98.45% with 49 synapses recorded


838410
Successfully loaded data from 838410_2026-03-02_12-40-55
% valid trials on DMD1: 100.0% with 57 synapses recorded
% valid trials on DMD2: 90.77% with 31 synapses recorded

Successfully loaded data from 838410_2026-03-03_13-49-07
% valid trials on DMD1: 94.3% with 91 synapses recorded
% valid trials on DMD2: 91.71% with 52 synapses recorded

Successfully loaded data from 838410_2026-03-04_12-54-47
% valid trials on DMD1: 96.89% with 99 synapses recorded
% valid trials on DMD2: 98.96% with 58 synapses recorded

Successfully loaded data from 838410_2026-03-05_10-16-37
% valid trials on DMD1: 94.3% with 71 synapses recorded
% valid trials on DMD2: 98.45% with 61 synapses recorded

Successfully loaded data from 838410_2026-03-18_16-43-23
% valid trials on DMD1: 96.37% with 95 synapses recorded
% valid trials on DM

In [8]:
# Summarize synapse yield by recording session day and imaging depth.
#
# Uses the same access pattern as the QC loop above:
#   1. Use `session_df` from summary.xlsx / sessions.
#   2. Restrict to `target_mice` and recording session_# 2-7.
#   3. Load the most recent SummaryLoCo*.mat for each session.
#   4. Read the number of synapses recorded on DMD1/DMD2 and map each DMD to
#      its depth using dmd1_depth / dmd2_depth from the sessions table.

session_day_col = 'session_#'
recording_days = range(2, 8)

target_session_df = session_df[
    session_df['subject_id'].isin(target_mice)
    & pd.to_numeric(session_df[session_day_col], errors='coerce').isin(recording_days)
].copy()

target_session_df[session_day_col] = pd.to_numeric(
    target_session_df[session_day_col], errors='coerce'
).astype('Int64')

session_depth_rows = []
missing_summary_rows = []

for _, sess_row in target_session_df.sort_values(['subject_id', session_day_col]).iterrows():
    mouse = int(sess_row['subject_id'])
    session_id = sess_row['session_id']
    session_day = int(sess_row[session_day_col])
    session_type = sess_row.get('session_type', None)
    session_dir = sess_row.get('session_dir', None)

    if pd.isna(session_dir):
        missing_summary_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_dir': session_dir,
            'reason': 'missing session_dir in summary.xlsx',
        })
        continue

    summary_paths = glob.glob(
        os.path.join(str(session_dir), '**', 'SummaryLoCo**.mat'),
        recursive=True,
    )

    if len(summary_paths) == 0:
        missing_summary_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_dir': session_dir,
            'reason': 'no SummaryLoCo*.mat found',
        })
        continue

    summary_loco_path = max(summary_paths, key=os.path.getmtime)

    try:
        exp = gs.GlutamateSummary(summary_loco_path)
    except Exception as exc:
        missing_summary_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_dir': session_dir,
            'reason': f'GlutamateSummary load failed: {exc}',
        })
        continue

    for dmd in (1, 2):
        depth_col = f'dmd{dmd}_depth'
        depth_um = sess_row.get(depth_col, np.nan)

        try:
            n_synapses = exp.n_synapses[dmd - 1]
        except (IndexError, TypeError):
            n_synapses = np.nan

        session_depth_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_type': session_type,
            'dmd': dmd,
            'depth_um': depth_um,
            'n_synapses': n_synapses,
            'summary_loco_path': summary_loco_path,
        })

session_depth_counts = pd.DataFrame(session_depth_rows)
missing_summary_df = pd.DataFrame(missing_summary_rows)

if session_depth_counts.empty:
    print('No SummaryLoCo files were loaded for the target mice / session_# 2-7 filter.')
else:
    session_depth_counts['depth_um'] = pd.to_numeric(
        session_depth_counts['depth_um'], errors='coerce'
    )
    session_depth_counts['n_synapses'] = pd.to_numeric(
        session_depth_counts['n_synapses'], errors='coerce'
    )

    session_depth_summary = (
        session_depth_counts
        .dropna(subset=['depth_um'])
        .groupby(['session_#', 'depth_um'], as_index=False)
        .agg(
            n_synapses=('n_synapses', 'sum'),
            n_mice=('subject_id', 'nunique'),
            n_sessions=('session_id', 'nunique'),
            n_dmds=('dmd', 'count'),
        )
        .sort_values(['session_#', 'depth_um'])
    )

    synapse_count_pivot = (
        session_depth_summary
        .pivot(index='session_#', columns='depth_um', values='n_synapses')
        .fillna(0)
        .astype(int)
    )

    mouse_count_pivot = (
        session_depth_summary
        .pivot(index='session_#', columns='depth_um', values='n_mice')
        .fillna(0)
        .astype(int)
    )

    print('Detailed DMD-level synapse counts:')
    display(session_depth_counts.sort_values(['session_#', 'depth_um', 'subject_id', 'dmd']))

    print('Aggregated counts by recording session day and depth:')
    display(session_depth_summary)

    print('Synapse count pivot: rows = recording session_#, columns = depth_um')
    display(synapse_count_pivot)

    print('Mouse contribution pivot: rows = recording session_#, columns = depth_um')
    display(mouse_count_pivot)

if not missing_summary_df.empty:
    print('Sessions skipped or not loaded:')
    display(missing_summary_df.sort_values(['session_#', 'subject_id']))


Detailed DMD-level synapse counts:


,subject_id,session_id,session_#,session_type,dmd,depth_um,n_synapses,summary_loco_path
12,803496,803496_2025-07-25_13-02-10,2,familiar,1,25,92,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
24,804730,804730_2025-07-25_14-08-35,2,familiar,1,25,101,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
36,804733,804733_2025-07-25_15-17-00,2,familiar,1,25,92,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
60,810196,810196_2025-07-25_16-24-20,2,familiar,1,25,133,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
72,826033,826033_2026-02-21_09-23-34,2,familiar,1,25,61,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
...,...,...,...,...,...,...,...,...
83,826033,826033_2026-02-27_13-53-35,7,novel+,2,200,43,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
94,834788,834788_2026-03-20_12-44-00,7,novel+,1,200,34,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
106,838410,838410_2026-03-20_10-00-59,7,novel+,1,200,63,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
11,803121,803121_2025-11-06_12-12-23,7,novel+,2,250,27,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...


Aggregated counts by recording session day and depth:


,session_#,depth_um,n_synapses,n_mice,n_sessions,n_dmds
0,2,25,614,7,7,7
1,2,100,214,4,4,4
2,2,200,202,5,5,5
3,2,250,88,2,2,2
4,3,25,506,7,7,7
5,3,100,163,4,4,4
6,3,200,278,5,5,5
7,3,250,70,2,2,2
8,4,25,549,7,7,7
9,4,100,227,4,4,4


Synapse count pivot: rows = recording session_#, columns = depth_um


depth_um,25,100,200,250
session_#,,,,
2,614,214,202,88
3,506,163,278,70
4,549,227,337,146
5,583,167,287,145
6,591,208,336,142
7,497,149,224,77


Mouse contribution pivot: rows = recording session_#, columns = depth_um


depth_um,25,100,200,250
session_#,,,,
2,7,4,5,2
3,7,4,5,2
4,7,4,5,2
5,7,4,5,2
6,7,4,5,2
7,7,4,5,2


In [46]:
session_depth_counts[session_depth_counts['subject_id']==834788].sort_values('depth_um')

,subject_id,session_id,session_#,session_type,dmd,depth_um,n_synapses,summary_loco_path
84,834788,834788_2026-03-03_09-22-19,2,familiar,1,25,44,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
86,834788,834788_2026-03-04_08-43-07,3,familiar,1,25,64,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
88,834788,834788_2026-03-05_08-11-16,4,familiar,1,25,87,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
90,834788,834788_2026-03-17_15-17-36,5,novel,1,25,73,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
93,834788,834788_2026-03-19_09-05-56,6,novel+,2,25,105,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
95,834788,834788_2026-03-20_12-44-00,7,novel+,2,25,49,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
85,834788,834788_2026-03-03_09-22-19,2,familiar,2,200,35,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
87,834788,834788_2026-03-04_08-43-07,3,familiar,2,200,37,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
89,834788,834788_2026-03-05_08-11-16,4,familiar,2,200,36,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
91,834788,834788_2026-03-17_15-17-36,5,novel,2,200,73,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
